# Replication

Artifact-level MNIST replication. This reads the compact paper-derived requirements table and rewrites the replication result CSV.

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/casparbreloh/rbt4dnn-seminar.git"
COLAB_ROOT = Path("/content/rbt4dnn-seminar")
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((path for path in candidates if (path / "data/requirements.csv").exists()), None)
if ROOT is None:
    ROOT = COLAB_ROOT
    if (ROOT / ".git").exists():
        subprocess.run(["git", "-C", str(ROOT), "fetch", "origin", "main"], check=True)
        subprocess.run(["git", "-C", str(ROOT), "reset", "--hard", "origin/main"], check=True)
    else:
        if ROOT.exists():
            shutil.rmtree(ROOT)
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)], check=True)

for module in [
    "shared",
    "mnist",
    "precondition_validity",
    "cost_analysis",
    "mnist_lora_ablation",
    "mnist_shared_generator",
]:
    sys.modules.pop(module, None)

EXPERIMENT = ROOT / "experiments" / "replication"
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(EXPERIMENT))
print(ROOT)
commit = subprocess.check_output(
    ["git", "-C", str(ROOT), "log", "-1", "--oneline"], text=True
).strip()
print(commit)

In [ ]:
from mnist import write_replication_summary
from shared import read_csv_rows

results_path = write_replication_summary(ROOT)
rows = read_csv_rows(results_path)

print("req | local n=100 | paper ref | delta")
for row in rows:
    print(
        f"{row['requirement']} | {row['local_pass_rate_n100']} | "
        f"{row['paper_pass_rate']} | {row['delta']}"
    )
print(results_path)

The local 100-image classifier check remains close to the paper-release reference values.